In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
import random
from urllib.parse import urljoin, urlparse

In [ ]:
visited = set()

def chunk_scraper(url, chunk_size=500):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = ' '.join(text.split()) 

    words = text.split()
    chunks, chunk = [], []
    length = 0
    for word in words:
        chunk.append(word)
        length += len(word) + 1
        if length >= chunk_size:
            chunks.append(' '.join(chunk))
            chunk, length = [], 0
    if chunk:
        chunks.append(' '.join(chunk))
    return chunks

def recursive_duke_scrape(url, data, base_domain="duke.edu", max_depth=2, depth=0):
    global visited
    if depth > max_depth or url in visited:
        return
    visited.add(url)

    print(f"\n[Depth {depth}] Visiting: {url}")
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return

    try:
        chunks = chunk_scraper(url)
        for c in chunks:
            data.append({"url": url, "content": c})
    except Exception as e:
        print(f"Failed to extract from {url}: {e}")
        return

    soup = BeautifulSoup(resp.text, "html.parser")
    links = soup.find_all("a")
    for a in links:
        href = a.get("href")
        if not href:
            continue

        full_url = urljoin(url, href)
        domain = urlparse(full_url).netloc

        if base_domain in domain and full_url not in visited:
            time.sleep(random.uniform(0, 10))
            recursive_duke_scrape(full_url, data, base_domain, max_depth, depth + 1)

start_url = "https://duke.edu/about/"
start_url = "https://pratt.duke.edu/"
start_url = "https://masters.pratt.duke.edu/ai/"
data = []
recursive_duke_scrape(start_url, data, max_depth=2)



[Depth 0] Visiting: https://masters.pratt.duke.edu/ai/

[Depth 1] Visiting: https://masters.pratt.duke.edu/

[Depth 2] Visiting: https://masters.pratt.duke.edu/apply

[Depth 2] Visiting: https://masters.pratt.duke.edu/admissions/

[Depth 2] Visiting: https://masters.pratt.duke.edu/apply/

[Depth 2] Visiting: https://masters.pratt.duke.edu/admissions/tuition-financial-aid/

[Depth 2] Visiting: https://masters.pratt.duke.edu/admissions/admitted/

[Depth 2] Visiting: https://masters.pratt.duke.edu/programs/

[Depth 2] Visiting: https://masters.pratt.duke.edu/programs/certificates/

[Depth 2] Visiting: https://masters.pratt.duke.edu/programs/degree-requirements/

[Depth 2] Visiting: https://masters.pratt.duke.edu/programs/options/

[Depth 2] Visiting: https://masters.pratt.duke.edu/life/

[Depth 2] Visiting: https://masters.pratt.duke.edu/life/career-services/

[Depth 2] Visiting: https://masters.pratt.duke.edu/life/students/

[Depth 2] Visiting: https://masters.pratt.duke.edu/news/

[Dep

In [15]:
df = pd.DataFrame(data)
print(len(df))
df.head()

1621


,url,content
0,https://masters.pratt.duke.edu/ai/,AI | Duke Engineering Master's Programs Apply ...
1,https://masters.pratt.duke.edu/ai/,immersive Master of Engineering in Artificial ...
2,https://masters.pratt.duke.edu/ai/,Leadership & Staff News Student Resources Requ...
3,https://masters.pratt.duke.edu/ai/,in-demand knowledge and skills 2 business cour...
4,https://masters.pratt.duke.edu/ai/,who wants to earn bachelor’s and master’s degr...


In [16]:
df.to_parquet('aipi.parquet')